# Generating positioned roughness elements

The roughness array is generated inside a bounding box and draped onto one or more
surfaces, so the elements follow the terrain profile instead of sitting on a flat plane.

Only the X and Y of the bounding box constrain the placement: the Z of each element comes
from the surface it is draped onto.

Read parameters from file

In [ ]:
import pathlib
import pprint

from cfdmod.roughness import PositionParams

pp = pprint.PrettyPrinter()

cfg_file = pathlib.Path("./fixtures/tests/roughness_gen/position_params.yaml")
cfg = PositionParams.from_file(cfg_file)

output_path = pathlib.Path("./output/roughness_gen")

pp.pprint(cfg.model_dump())

Generate and position the elements.

`position_pattern` reads the surfaces, clips the bounding box to their extent, fills it
with the element array and lifts every element onto the surface below it. Elements whose
base falls outside every surface are dropped.

In [ ]:
from cfdmod.roughness import position_pattern

triangles, normals = position_pattern(
    element_params=cfg.element_params,
    spacing_params=cfg.spacing_params,
    bounding_box=cfg.bounding_box,
    surfaces=[pathlib.Path(p) for p in cfg.surfaces.values()],
)

print(f"Elements: {len(triangles) // 2}")
print(f"Triangles: {len(triangles)}")
print(f"Base Z range: {triangles[:, :, 2].min():.2f} to {triangles[:, :, 2].max():.2f}")

The surfaces can also be passed in memory (an `LnasFormat`, an `LnasGeometry` or a plain
`(N, 3)` vertex array), which avoids a write plus read for a caller that already holds the
mesh. The number of surface points fed to the interpolation is bounded by `max_points`;
pass `None` to use every vertex.

Export generated geometry

In [ ]:
from cfdmod.io.geometry.STL import export_stl

export_stl(output_path / "positioned_elements.stl", triangles, normals)